[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/semiconductor_devices/solar_cell.ipynb)

# A Solar Cell's Fingers

This example computes a solar cell in cross-section by drift and diffusion, between the thin metal lines that collect its current from its front, its fingers, and finds how far apart they are best laid. Light enters the silicon through the front and is absorbed as it goes deeper, making electrons and holes in pairs. Just beneath the surface a thin top layer, heavily doped with acceptors, meets the thick base, doped with donors, at a junction whose field takes the pairs apart: the holes the light makes in the base diffuse up to the junction and cross into the top layer, and there they flow sideways to the nearest finger, while their electrons flow down to a metal contact covering the back. The fingers are opaque and shade the silicon beneath them, so fingers further apart let in more of the light; but the top layer is thin, and its resistance costs voltage on the holes' way sideways, more the further apart the fingers lie, since the further the holes go along it, the more of them there are. The power the cell delivers peaks at a spacing between the two.

The algebra is the cross-section's plane in spacetime joined with four directions that have no metric, for the electrons, the holes, the heat and the light, `Algebra("x+y+t-") * Algebra("n0p0q0l0")`, and the drift, the diffusion and the field are the lines that compute a bar's edges and a transistor's triangles alike: each carrier is pushed by the potentials' drive, `-(gradients ^ potentials)`, and what it carries is gathered into each cell's outflow by the drive's adjugate. The light's generation of pairs is a field over the vertices, falling with the depth that the plane `mv.y` reads off each, `((mv.y & vertices) / absorption_length).exp()`, and nothing in the fingers' shadow. A hole drifts along the force the drive puts on it, `drive.commutator(hole)`, a direction of the plane, which the pictures streak. The period is mirrored at its sides with nothing imposed there: a cell's outflow gathers only the triangles about it, so that no flux leaves through a side with no triangle beyond it. Absorption is recombination run backward, its energy drawn from the light, `pairs - gap * light`, and the light's free energy at the sun's temperature bounds the voltage.

In the notation of photovoltaics, with $\rho_\square$ the top layer's sheet resistance, $S$ the fingers' spacing, $w$ their width, and $J_{mp}$ and $V_{mp}$ the current density and the voltage at the maximum power point, the top layer's resistance takes the share $\rho_\square J_{mp} S^2/(12\,V_{mp})$ of the power and the fingers' shadow the share $w/S$, and their sum is least at $S^3 = 6\,w V_{mp}/(\rho_\square J_{mp})$. The drift–diffusion model beneath is Poisson's equation $-\nabla\cdot(\epsilon\nabla\psi) = q(p - n + N_D^+ - N_A^-)$ with the currents $\mathbf J_n = \mu_n n\nabla E_{Fn}$ and $\mathbf J_p = \mu_p p\nabla E_{Fp}$ and their continuity, here with the light's generation $G = G_0\,e^{-z/L_\alpha}$ outside the shadow.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import math
from collections.abc import Iterator
from functools import partial, reduce
from operator import and_

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import SparseExtensor, spdiag
from examples.util.animation import show_states
from examples.thermodynamics.semiconductor_devices import render, scenarios

# The cross-section's plane in spacetime, and the electrons, the holes, the heat and the light a cell holds or trades:
# each a direction of its own, degenerate, since nothing measures an amount against a length or a time.
ga = Algebra("x+y+t-") * Algebra("n0p0q0l0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
# The planes reading a place across and up; the potentials, the electric potential along t and the carriers' chemical
# potentials along n and p; and their drive, a slope across the plane wedged with what each acts on.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Potentials = ga.gatype.from_blades("t n p")                                   # Vector+
Drive = ga.gatype.from_blades("xt yt xn yn xp yp")                            # Bivector+
# A unit of charge, along time; an electron, a unit of the electrons' count with a negative charge, and a hole, a unit
# of the holes' count with a positive one; a unit of heat, and of the light's energy.
time = -mv.t.dual()                                                           # [] Point-
electron = mv.n.dual() - time                                                 # [] Point-
hole = mv.p.dual() + time                                                     # [] Point-
heat = mv.q.dual()                                                            # [] Point-
light = mv.l.dual()                                                           # [] Point-
# An electron and a hole, what recombination takes and absorption makes.
pairs = electron + hole                                                       # [] Point-
# The carriers; their counts, which their chemical potentials read, the planes that read each count off an amount,
# and the carriers' charges, along time.
carriers = stack((electron, hole))                                            # [carriers] Point-
counts = stack((mv.n.dual(), mv.p.dual()))                                    # [carriers] Point-
counting = stack((mv.n, mv.p))                                                # [carriers] Plane+
charges = carriers - counts                                                   # [carriers] Point-

## The cross-section

One period of the cell runs from the middle of one finger to the middle of the next, mirrored at its sides, since the cell repeats from finger to finger. Its base, doped with donors, is 160 Debye lengths deep, and its top layer, doped six times as heavily with acceptors, 8; a thin layer at the back, far more heavily doped with donors than the base, keeps the holes off the metal contact covering the back, which would otherwise take those the light makes deepest. The fingers are 16 Debye lengths wide. Lengths are in the base's Debye length, about 40 nm at 10¹⁶ donors a cubic centimetre; energies and potentials are in the thermal energy, about 26 meV; densities are in the base's doping; and times are in the time an electron takes to diffuse across a Debye length. A real cell's base is some thirty times deeper and its fingers a hundred times or more further apart; scaled so, the base, the top layer and the fingers' spacing show in one picture. The mesh is a grid of triangles, finest through the top layer and the junction beneath it, its rows the same at every spacing of the fingers and its columns as many, stretched across the gap between two fingers.

In [ ]:
# Silicon at room temperature: its intrinsic density, a millionth of the base's doping; the band gap in thermal units,
# 1.12 eV; the electrons' and the holes' mobilities; the permittivity; and the rate constant of recombination, one over
# the base's doping times the holes' lifetime there, long enough for them to diffuse further than the base is deep.
intrinsic, gap = 1e-6, 43.6
mobilities, permittivity = np.array([1.0, 1 / 3]), 1.0
rate_constant = 1 / 2e5
# The light: the pairs it makes per area and time just beneath the surface, and the depth over which that falls by a
# factor e.
surface_generation, absorption_length = 1.5e-4, 50.0

In [ ]:
# The fingers' spacings to be swept, and the cell's period at the first: its triangles, its vertices' places across
# and up, the net doping at each vertex, the donors less the acceptors, whether each vertex lies in a finger's shadow,
# and the vertices of its contacts, the fingers and the back.
finger_spacings = scenarios.finger_spacings                                   # [spacings]
topology, coordinates, doping, shaded, contacts = scenarios.period_grid(finger_spacings[0])
vertices = mv(Upright, coordinates).dual().field()                            # Point-[vertices]
dopants = (time * doping).field()                                             # Point-[vertices]
fingers, back = contacts
held = np.concatenate(contacts)                                               # [held]
# The light comes straight down through the surface, where the plane y reads nothing, and falls by a factor e over
# each absorption length below it; the fingers shade the silicon beneath them.
falling = surface_generation * ((mv.y & vertices) / absorption_length).exp()   # Scalar+[vertices]
generation = mv.scalar(np.where(shaded, 0.0, 1.0)[:, None]).field() * falling   # Scalar+[vertices]
render.draw_cell(topology, vertices, dopants, contacts, generation, finger_spacings[0]);

The cell at the smallest spacing of the fingers swept, its period repeated across the picture: the fingers dark on the surface, the top layer a thin rose band beneath them and the back layer a blue one above the back contact, and the silicon tinted by the light's generation, warm at the surface and fading with depth, and white in the fingers' shadow.

## Drift, diffusion and the field

A field varies across each triangle by its corners' hats. A corner's hat gradient is the side facing it joined with time and the amounts, which a field over the plane does not vary along, over the same join closed by the step to the corner; the drive across each triangle is then one map of the potentials at its corners, `driving`. Read backwards across the regressive product, the drive's adjugate gathers the fluxes in the triangles into what flows out of each vertex's cell, its amounts read from the left as the potentials read them, `driving.adjugate().involute()`. The fingers' spacing moves the vertices, so these maps are made for each spacing from the vertices alone.

A steady state balances, in every cell, what the carriers' fluxes take out against what recombines and what the light makes; Gauss's law, the field's outflow from each cell against the charge it holds, joins the carriers' balance along time, where that balance only repeats itself. Newton's method solves it, each carrier's balance taken per carrier, so that carriers far scarcer than the plentiful ones weigh alike in the solve.

In [ ]:
corner_mean = topology.incidence[-1, 0].mean(axis=0)                          # Scalar+[simplices <- vertices]
# Time and every amount, joined, which a field over the plane does not vary along.
unread = time & counts[0] & counts[1] & heat & light                          # [] Span-


def simplices(vertices: Point) -> tuple[Scalar, Scalar, SparseExtensor, SparseExtensor]:
    """Each triangle's area `Scalar+[simplices]`, each vertex's cell, a share of the triangles about it
    `Scalar+[vertices]`, the drive across each triangle of the potentials at its corners
    `(Drive+ <- Potentials+)[simplices <- vertices]`, and what flows out of each vertex's cell of what flows in the
    triangles about it `(Point- <- Flux-)[vertices <- simplices]`, for the triangles at these vertices."""
    corners = topology.incidence[-1, 0] * vertices                            # [corners] Point-[simplices]
    # The corners of the side facing each corner, the first of them a step from the corner.
    facing = corners[topology.complements[-1, 0]]                             # [side corners, corners] Point-[simplices]
    steps = corners - facing[0]                                               # [corners] Point-[simplices]
    # The side facing each corner joined with time and the amounts; over the same join closed by the step to the
    # corner, it is the gradient that reads one from the side to the corner.
    facets = reduce(and_, facing[1:] - facing[0], unread)                     # [corners] Plane-[simplices]
    gradients = facets / (facets & steps)                                     # [corners] Plane+[simplices]
    # The integrals weigh by each triangle's area, an ordinary number: the join of its edges from its first corner, with
    # time and the amounts, over the factorial of its dimension.
    joins = reduce(and_, corners[1:] - corners[0], unread)                    # Scalar-[simplices]
    volumes = joins.norm() / math.factorial(topology.n_dim)                   # Scalar+[simplices]
    # The drive: minus the potentials' slope, their corners' values on their hats' gradients, wedged with what each
    # acts on; read backwards, its amounts read from the left, the outflow.
    slopes = (spdiag(gradients) * topology.incidence[-1, 0]).sum(axis=0)      # Plane+[simplices <- vertices]
    driving = -(slopes ^ Potentials)                                          # (Drive+ <- Potentials+)[simplices <- vertices]
    return volumes, ~corner_mean * volumes, driving, driving.adjugate().involute()

In [ ]:
# A carrier drifts along the force a drive puts on it, and its flux is the carrier swept along that drift: each
# carrier's conductance per unit of its density, the same in every direction. A unit of charge swept so, times the
# permittivity, is the field's displacement.
conducting = mobilities * Drive.commutator(carriers).regressive(carriers)     # [carriers] Flux- <- Drive+
displacement = permittivity * Drive.commutator(time).regressive(time)         # [] Flux- <- Drive+


def carrier_densities(potentials: Potentials) -> Scalar:
    """The electrons and the holes per area at these potentials `[carriers] Scalar+[vertices]`: the intrinsic density
    times the exponential of each carrier's chemical potential, Boltzmann's distribution."""
    return intrinsic * (potentials & counts).exp()


def steady(volumes: Scalar, cells: Scalar, driving: SparseExtensor, outflow: SparseExtensor, potentials: Potentials,
           contacts: Potentials, generations: Scalar) -> Iterator[Potentials]:
    """The steady potentials `Potentials+[vertices]` at each frame, from these, in the triangles of these areas and
    cells, drives and outflows: the contacts' vertices held at that frame's potentials `[frames] Potentials+[held]`,
    the light making that frame's generation `[frames] Scalar+[vertices]`, each solved by Newton's steps until one
    changes no chemical potential by more than the tolerance."""
    holding = topology.reading(held)                                          # Scalar+[held <- vertices]
    free = topology.reading(np.setdiff1d(np.arange(topology.n_vertices), held))   # Scalar+[free <- vertices]
    for contact, generation in zip(contacts, generations):
        # The contacts take their potentials; the rest start from where they were.
        potentials = ~free * (free * potentials) + ~holding * contact         # Potentials+[vertices]
        while True:
            densities = carrier_densities(potentials)                         # [carriers] Scalar+[vertices]
            drive = driving(potentials)                                       # Drive+[simplices]
            conductance = ((corner_mean * densities) * conducting).sum(axis=0)   # (Flux- <- Drive+)[simplices]
            # Recombination by mass action, less the light's generation, pairs per area and time.
            rates = rate_constant * (densities[0] * densities[1] - intrinsic ** 2) - generation   # Scalar+[vertices]
            # Gauss's law, along time: the field's outflow from each cell against the charge it holds.
            charge = (densities * charges).sum(axis=0) + dopants              # Point-[vertices]
            gauss = outflow(volumes * displacement(drive)) - cells * charge  # Point-[vertices]
            # What the carriers' fluxes take out of each cell and what recombines there; along time this only repeats
            # the electrons' and the holes' balance, which Gauss's law completes there.
            residual = outflow(volumes * conductance(drive)) + cells * rates * pairs + gauss   # Point-[vertices]
            # Newton's map: the charge's capacitance in Gauss's law; the response of the carriers' fluxes and of the
            # field through the drive, and of the fluxes through the densities on each triangle, by each carrier's
            # flux per unit of its density, gathered into each cell's outflow; and recombination's, by the pairs' dyad.
            capacitance = spdiag(cells * (densities * -charges * (Potentials & counts)).sum(axis=0))
            through_drive = spdiag(volumes * (conductance + displacement))(driving)
            through_densities = (spdiag(volumes * conducting(drive)) * corner_mean
                                 * spdiag(densities * (Potentials & counts))).sum(axis=0)
            recombining = spdiag(cells * rate_constant * densities[0] * densities[1] * (pairs * (Potentials & pairs)))
            newton = capacitance + recombining + outflow(through_drive + through_densities)
            # Each carrier's balance taken per carrier, over its density, the charge's as it is.
            per_carrier = spdiag(time * (mv.t & Point) + (counts * (counting & Point) / densities).sum(axis=0))
            solving = free * per_carrier(newton) * ~free                      # (Point- <- Potentials+)[free <- free]
            change = -(~free * solving.solve(free * per_carrier(residual)))   # Potentials+[vertices]
            # Newton's step scaled down at each vertex where it would change the chemical potentials there by much
            # more than the limit: the densities follow them exponentially, and a full step from far off overshoots by
            # powers of e. The residual is linear in the electric potential, whose change needs no holding back.
            changes = (counts & change).abs().sum(axis=0)                     # Scalar+[vertices]
            potentials = potentials + change / (1 + changes / limit)
            if changes.max() < tolerance:
                break
        yield potentials

## The light and the bias

The contacts hold the potentials at their vertices, ohmic, each at neutral silicon of its doping, with its Fermi level at nothing: the fingers on the top layer, their electric potential raised by the bias, and the back contact on the back layer. The cell is found first in the dark at no bias; then the light comes on at no bias, the contacts shorted, and the bias rises to open circuit, ever more slowly as the current falls away. The current the cell delivers is the charge its carriers' fluxes take out of the back's cells, `mv.t & outflow(volumes * flux)` there, per length of the cross-section's depth. The cells below take about forty seconds, most of it solving for the steady states.

In [ ]:
# Newton's steps: how far one may change a chemical potential, in thermal units, and how little the last one changes
# them; the frames' duration in milliseconds.
limit, tolerance = 2.0, 1e-3
frame_duration_ms = 40
# The bias at each frame, in thermal units, and the share of the light: in the dark at no bias, the light coming on
# at no bias, and the bias rising to open circuit; the volt in thermal units, at 300 K.
biases, shares = scenarios.lighting_biases()                                  # [frames], [frames]
volt = 1 / 0.02585

In [ ]:
# Neutral silicon at the doping with the Fermi level at nothing: each carrier's electrochemical potential is nothing,
# so that the electric potential is the electrons' chemical potential.
chemical = scenarios.slabs(doping)                                            # [carriers, vertices]
neutral = (mv.t * chemical[0] + mv.n * chemical[0] + mv.p * chemical[1]).field()   # Potentials+[vertices]


def holding_at(biases: np.ndarray) -> Potentials:
    """The potentials the contacts hold at each of these biases `[frames]`, `[frames] Potentials+[held]`: neutral
    silicon's at their doping, the bias raising the fingers' electric potential."""
    raised = np.concatenate([np.repeat(biases[:, None], len(fingers), 1), np.zeros((len(biases), len(back)))], axis=-1)
    return topology.reading(held) * neutral + (mv.t * raised).field()


def terminal(volumes: Scalar, driving: SparseExtensor, outflow: SparseExtensor,
             potentials: Potentials) -> tuple[Scalar, Scalar]:
    """The cell's voltage at these potentials `[] Scalar+`, the rise of the electrons' quasi-Fermi level from the
    fingers to the back, and the current it delivers `[] Scalar+`, the charge the carriers' fluxes take out of the
    back's cells, in the triangles of these areas, drives and outflows."""
    levels = potentials & electron                                            # Scalar+[vertices]
    rise = (topology.reading(back) * levels).sites.mean() - (topology.reading(fingers) * levels).sites.mean()
    conductance = ((corner_mean * carrier_densities(potentials)) * conducting).sum(axis=0)   # (Flux- <- Drive+)[simplices]
    outflowing = outflow(volumes * conductance(driving(potentials)))          # Point-[vertices]
    return rise, (topology.reading(back) * (mv.t & outflowing)).sites.sum()


def holes_flow(driving: SparseExtensor, potentials: Potentials) -> Point:
    """The holes' flow in each triangle at these potentials `Point-[simplices]`, for these drives: the force the drive
    puts on a hole, the direction it drifts along, times its mobility and its density there."""
    densities = corner_mean * carrier_densities(potentials)[1]                # Scalar+[simplices]
    return mobilities[1] * densities * driving(potentials).commutator(hole)


volumes, cells, driving, outflow = simplices(vertices)
# The streaks' scale, the holes' flow beneath a surface where the light makes all its pairs above; and the current's,
# the pairs the light makes in the period each time, which the junction gathers nearly all of.
reference_flow = surface_generation * absorption_length
made = (cells * generation).sites.sum()                                       # [] Scalar+
lit = show_states(steady(volumes, cells, driving, outflow, neutral, holding_at(biases), shares * generation),
                  render.animate_lighting, topology, vertices, dopants, contacts, generation,
                  partial(holes_flow, driving), partial(terminal, volumes, driving, outflow), reference_flow,
                  finger_spacings[0], biases.max(), made, volt, shares, name="solar_cell_lighting",
                  duration_ms=frame_duration_ms)

The cell at the smallest spacing, several fingers in view, beside the current it delivers against its voltage, in volts. The streaks follow the holes, longer where their flow is stronger, as the square root of it. In the dark nothing flows. As the light comes on, the holes it makes rise through the base to the junction, cross into the top layer and run along it to the fingers, the current growing with the light; beneath each finger, where the light makes none, holes from the lit silicon on either side rise to the junction too. The electrons the light makes with them flow down to the back. As the bias rises the current barely falls at first, and near the open-circuit voltage, about 0.68 V, it falls away to nothing while the holes go on rising as before: at open circuit the junction's own forward current returns as many carriers as the light makes.

## The fingers' spacing

The fingers spread from a few of their widths apart to several of the base's depths, the same share further at each frame, and at each spacing the cell is held at its maximum power point, which moves little from one spacing to the next. Each spacing's steady states start from the last spacing's, its vertices moved, since the grid is the same. The cells below take about a minute, most of it solving for the steady states.

In [ ]:
# How far apart the biases about each spacing's maximum lie, in thermal units; the first spacing's maximum, between
# the frames of the sweep of the bias, and the most its surface would deliver unshaded, its power per length of the
# surface the light falls on, which the power chart reaches a little past.
bias_step = 1.5
lit_readings = [terminal(volumes, driving, outflow, state) for state in lit]
lit_powers = stack([voltage * current for voltage, current in lit_readings]) / finger_spacings[0]   # [frames] Scalar+
best, best_bias = scenarios.peak(biases, lit_powers)
unshaded = lit_powers[best] * finger_spacings[0] / (finger_spacings[0] - scenarios.finger_width)   # [] Scalar+

In [ ]:
def spreading(potentials: Potentials, bias: float) -> Iterator[tuple[Point, Potentials]]:
    """The cell at its maximum power at each spacing of the fingers in turn, its vertices `Point-[vertices]` and its
    steady potentials `Potentials+[vertices]`, from these near the first spacing's maximum at this bias: at each
    spacing its steady states at the last maximum's bias and a step either side of it, the middle one the spacing's
    own, and the peak of the parabola through their powers the next spacing's bias."""
    for finger_spacing in finger_spacings:
        vertices = mv(Upright, scenarios.cell_places(finger_spacing)).dual().field()   # Point-[vertices]
        volumes, cells, driving, outflow = simplices(vertices)
        trials = bias + bias_step * np.array([-1.0, 0.0, 1.0])                # [trials]
        states = list(steady(volumes, cells, driving, outflow, potentials, holding_at(trials),
                             np.ones(3) * generation))
        readings = [terminal(volumes, driving, outflow, state) for state in states]
        _, bias = scenarios.peak(trials, stack([voltage * current for voltage, current in readings]))
        potentials = states[1]
        yield vertices, potentials


def holes_at(vertices: Point, potentials: Potentials) -> Point:
    """The holes' flow in each triangle `Point-[simplices]` at these vertices and potentials."""
    return holes_flow(simplices(vertices)[2], potentials)


def power_at(finger_spacing: float, vertices: Point, potentials: Potentials) -> Scalar:
    """The power the cell delivers per length of its surface `[] Scalar+`, its fingers this far apart, at these
    vertices and potentials: its voltage times its current, over the period."""
    volumes, _, driving, outflow = simplices(vertices)
    voltage, current = terminal(volumes, driving, outflow, potentials)
    return voltage * current / finger_spacing


spread = show_states(spreading(lit[best], best_bias), render.animate_spreading, topology, dopants, contacts,
                     generation, holes_at, power_at, reference_flow, finger_spacings, unshaded,
                     name="solar_cell_spacing", duration_ms=frame_duration_ms)

The fingers spreading in a window of the cell, beside the power per length of surface against their spacing, on a logarithmic scale. Close together, the fingers shade much of the surface, and the holes reach them after a short way along the top layer. As they spread, less of the light falls on metal and the power rises; but the holes run ever further along the thin top layer, crowding toward each finger, and the voltage the top layer costs them grows. Past the peak that cost wins and the power falls. At the widest spacings the base itself carries holes sideways toward the finger, its streaks fanning in from both sides beneath the far gaps.

## The best spacing

The power the cell delivers peaks between the shade and the resistance, and the period at the peak shows where the holes go there.

In [ ]:
powers = stack([power_at(finger_spacing, *state) for finger_spacing, state in zip(finger_spacings, spread)])   # [spacings] Scalar+
optimum = powers.argmax()
render.draw_power(finger_spacings, powers);
print(f"the best spacing: {finger_spacings[optimum] / scenarios.finger_width:.1f} fingers' widths")

The power per length of the cell's surface at its maximum power point against the fingers' spacing, on a logarithmic scale, the dot at its peak: closer, the shade costs more than the resistance saves, and further, the resistance costs more than the light it lets in.

In [ ]:
optimal_vertices, optimal_potentials = spread[optimum]
render.draw_period(topology, optimal_vertices, dopants, contacts, generation,
                   holes_at(optimal_vertices, optimal_potentials), reference_flow, finger_spacings[optimum]);

One period at the best spacing, from the middle of one gap to the middle of the next, its finger in the middle. The holes the light makes rise to the junction and run along the top layer to the finger from both sides; beneath the finger, in its shadow, holes from the lit silicon on either side rise to the junction too, and deep down, where little of the light arrives, the flow is faint.

In [ ]:
# checks
# Under the fingers the light makes no pairs: none in the cells of the vertices in their shadow.
shadow = topology.reading(np.flatnonzero(shaded))                             # Scalar+[shaded <- vertices]
assert (shadow * (cells * generation)).sites.sum().to_array() == 0
# Further apart, the fingers shade less of the surface: the light makes more pairs per length of it.
made_per_length = stack([(simplices(vertices)[1] * generation).sites.sum() / finger_spacing
                         for finger_spacing, (vertices, _) in zip(finger_spacings, spread)])   # [spacings] Scalar+
assert np.all(np.diff(made_per_length.to_array()) > 0)
# Further apart, the top layer costs more voltage: the holes' quasi-Fermi level spreads further along the surface,
# from the finger's edge, the last of the surface's vertices beneath it, to the middle of the gap.
surface = np.flatnonzero(coordinates[:, 1] == 0)                              # [columns]
edge, middle = surface[len(fingers) // 2 - 1:len(fingers) // 2], surface[len(surface) // 2:len(surface) // 2 + 1]
across_gap = topology.reading(middle) - topology.reading(edge)                # Scalar+[1 <- vertices]
spreads = np.abs(stack([(across_gap * (potentials & hole)).sites.sum() for _, potentials in spread]).to_array())
assert np.all(np.diff(spreads) > 0) and spreads[-1] > 30 * spreads[0]
# Between the two the power peaks, well above its value at either end of the sweep.
power_values = powers.to_array()                                              # [spacings]
assert 0 < optimum < len(power_values) - 1 and 10 < finger_spacings[optimum] / scenarios.finger_width < 20
assert power_values[0] < 0.75 * power_values[optimum] and power_values[-1] < 0.75 * power_values[optimum]
# At open circuit, absorption produces entropy everywhere: the energy plane pairs with it to nothing, and the
# entropy's slope, the energy plane less the potentials, the light's free energy its energy less the share the sun's
# temperature takes along, to more than nothing; the quasi-Fermi levels part by less than the Carnot bound.
sun = 5800 / 300
absorption = pairs - gap * light                                              # [] Point-
open_circuit = lit[-1]                                                        # Potentials+[vertices]
energy = (open_circuit & time) * mv.t + gap / 2 * (mv.n + mv.p) + mv.q + mv.l   # Plane+[vertices]
entropy_slope = energy - open_circuit - (1 - 1 / sun) * mv.l                  # Plane+[vertices]
np.testing.assert_allclose((energy & absorption).to_array(), 0.0, atol=1e-12)
assert np.all((entropy_slope & absorption).to_array() > 0)